<a href="https://colab.research.google.com/github/vibha-sanghani/EDA-Play-Store-App-Review-Analysis/blob/main/Vibha_EDA_Play_Store_App_Review_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name** - **Play Store App review Analysis**



##### **Project Type**    - EDA
##### **Contribution**    - Partnership
**Partner_1 - Name**: Kartikey Rastogi
**Partner_2 - Name**: Vibha Sanghani

# **Project Summary -**

Write the summary here within 500-600 words.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


**Business Context**

The Play Store apps data has enormous potential to drive app-making businesses to success. Actionable insights can be drawn for developers to work on and capture the Android market. Each app (row) has values for category, rating, size, and more. Another dataset contains customer reviews of the android apps. Explore and analyse the data to discover key factors responsible for app engagement and success.

#### **Define Your Business Objective?**

To find key factors responsible for app engagement and success.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
import seaborn as sns
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

### Dataset Loading

In [ ]:
# Load Dataset
play_store_data = pd.read_excel('Play Store Data.xlsx')
user_review_data = pd.read_excel('User Reviews.xlsx')

In [ ]:
# Creating the working dataframe of our given datasets so as to maintain the original dataset

play_store_df = pd.DataFrame(play_store_data)
user_review_df = pd.DataFrame(user_review_data)

### Dataset First View

**First we'll be working with the Play Store Dataset**

In [ ]:
# Dataset First Look
play_store_df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
play_store_df.shape

As we can see here this dataset has:-
- 10841 rows, and
- 13 columns

### Dataset Information

In [ ]:
# Dataset Info
play_store_df.info()

As we can see here the data types are as follows:-
dtypes: float64(1), object(12)
However, when we look at our dataset we can see that few of the features are having numerica values hence we need to correct these data types in the correct order.
We need to correct the datatypes as follows:-
- Reviews: int
- Installs: int
- Price: float
- Last Updated: datetype

In [ ]:
for value in play_store_df['Reviews']:
  if value == '3.0M':
    play_store_df['Reviews'] = play_store_df['Reviews'].replace('3.0M', '3000000')
play_store_df["Reviews"] = play_store_df["Reviews"].astype(int)

In [ ]:
try:
  for i in range(len(play_store_df['Installs'])):
    if play_store_df['Installs'][i] == 'Free':
      play_store_df['Installs'][i] = 0
    elif type(play_store_df['Installs'][i]) == str and play_store_df['Installs'][i][-1] == '+':
      play_store_df['Installs'][i] = play_store_df['Installs'][i][0:-1]
except IndexError as e:
  print(i)
play_store_df["Installs"] = play_store_df["Installs"].replace(',', '')
play_store_df["Installs"] = play_store_df['Installs'].fillna(0)
# play_store_df["Installs"] = play_store_df["Installs"].astype(int)
# Convert 'Installs' column to integer
play_store_df['Installs'] = play_store_df['Installs'].astype(int)


In [ ]:
for i in range(len(play_store_df["Price"])):
  if play_store_df['Price'][i] == 'Everyone':
    play_store_df['Price'][i] = 0
play_store_df["Price"] = play_store_df["Price"].astype(float)

In [ ]:
play_store_df.info()

Now we have successfully updated the data types of the concerned columns.

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
play_store_df.duplicated().value_counts()

As we can see 483 values are direct duplicates.

In [ ]:
# Let check the duplicated values
play_store_df.loc[play_store_df.duplicated()].head()

In [ ]:
# Let's remove these duplicated rows
play_store_df = play_store_df.drop_duplicates(keep='first').reset_index(drop=True)

In [ ]:
play_store_df.duplicated().value_counts()

The direct duplicates has been removed however we need to check if there are any other duplicates in our dataset with few different values.

In [ ]:
play_store_df.loc[play_store_df[["App", "Last Updated"]].duplicated()]

As we can see that there are 654 values that are duplicated however some values are different. Let us sort the values with the last updated date and keep the latest ones.

In [ ]:
# Last convert Last Updated columne to date type.
play_store_df['Last Updated'] = pd.to_datetime(play_store_df['Last Updated'], errors='coerce').dt.date

In [ ]:
# Sorting the values as per latest dates so that we can drop the duplicates and keep the first
play_store_df = play_store_df.sort_values(by='Last Updated', ascending=False).reset_index(drop=True)

In [ ]:
# Dropping the duplicate values in our dataset and keeping the latest ones
play_store_df.drop_duplicates(subset=['App', 'Last Updated'], keep='first', inplace=True)

In [ ]:
play_store_df.shape

In [ ]:
# Let's verify
play_store_df[["App", "Last Updated"]].duplicated().value_counts()

We have successfully droped the duplicates from our dataset.

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
play_store_df.isna().sum()

- As the rating of 1464 values is missing we can simply change these values to 0, being a big number it will be better to keep these values.
- We can alse see that 1 App name is also missing hence we need to drop it as we do not know which app it is.
- For the type missing value we will change it to 'Free'.
- For the content rating we can first check the app then assign the value.
- For the Versions we can simply change the values to 0.

In [ ]:
play_store_df['Rating'].fillna(0, inplace=True)
play_store_df['App'].dropna(inplace=True)
play_store_df['Type'].fillna('Free', inplace=True)
play_store_df['Current Ver'].fillna(0, inplace=True)
play_store_df['Android Ver'].fillna(0, inplace=True)

In [ ]:
play_store_df.loc[play_store_df['Content Rating'].isna()]

As it is a photoframe, the content rating can be for Everyone.

In [ ]:
play_store_df['Content Rating'].fillna('Everyone', inplace=True)
play_store_df.dropna(subset=['App'], inplace=True)
play_store_df.isna().sum()

We have successfully handled the missing values.

### What did you know about your dataset?

We have been provided with 2 datasets: -
- Play Store Data
- User Reviews Data

Play Store Dataset has 10841 rows, and 13 columns.
The datatype of few columns was incorrect, hecne we updated it as per below:-
  -	Reviews: int
  -	Installs: int
  -	Price: float
  -	Last Updated: datetype

After updating the data types, we checked the duplicated values and we found: -
  -	In the overall dataset there were 483 duplicates.
  - When we checked within the data taking “App” and “Last Updated” columns     
  there were 654 duplicate values.

We were having few NA values and we handled those values accordingly.
  -	Replaced 1464 NA values in Ratings with 0
  -	Drop 1 missing value in App column
  -	Replaced the missing values in Type column to Free.
  -	Checked the missing value in content rating as it is for a photoframe we changes it to Everyone.
  - 	Replaced the missing values of version with 0.

Our dataset has:-
- 4 Numerical Columns
- 9 Categorical Columns

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
play_store_df.columns

In [ ]:
# Dataset Describe
play_store_df.describe()

### Variables Description

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Before starting our data wrangling let us save our cleaned data into a different excel file so that we can save all our work in it's sheets
play_store_df.to_excel('cleaned_data.xlsx', index=False)

In [ ]:
# Write your code to make your dataset analysis ready.
# Let's check the top 10 Apps with the maximum installs

# Creating a new dataframe to check these values
app_install_df = pd.DataFrame({
    'App': play_store_df['App'],
    'Installs': play_store_df['Installs']
})

app_install_df.sort_values(by='Installs', ascending=False, inplace=True)
app_install_df.head(10)


In [ ]:
app_install_df.reset_index(drop=True)

### What all manipulations have you done and insights you found?

Answer Here.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
df= pd.read_excel('cleaned_data.xlsx')
df

sns.boxplot(data=df,x='App',y='Rating')
plt.show()

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 2

In [ ]:
# Chart - 2 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 3

In [ ]:
# Chart - 3 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 4

In [ ]:
# Chart - 4 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 5

In [ ]:
# Chart - 5 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 6

In [ ]:
# Chart - 6 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 7

In [ ]:
# Chart - 7 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 8

In [ ]:
# Chart - 8 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 9

In [ ]:
# Chart - 9 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 10

In [ ]:
# Chart - 10 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 11

In [ ]:
# Chart - 11 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 12

In [ ]:
# Chart - 12 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 13

In [ ]:
# Chart - 13 visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Answer Here

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code

##### 1. Why did you pick the specific chart?

Answer Here.

##### 2. What is/are the insight(s) found from the chart?

Answer Here

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

Answer Here.

# **Conclusion**

Write the conclusion here.

### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***